# Jour 1 · Variables, cible et types de problèmes

> **Version corrigée** — les cellules de solution sont signalées.

## Objectifs

- identifier observation, feature et cible
- distinguer classification, régression et non supervisé
- construire X et y sans inclure d’information interdite


## Partir de la question métier

Un projet ML commence par une question dont la réponse est observable. « Anticiper les problèmes » est trop vague. « Une panne surviendra-t-elle dans les sept prochains jours ? » définit une réponse binaire et une période précise.

Une **observation** est un cas individuel : ici une machine à un instant de référence. Une **feature**, ou variable d'entrée, est une information disponible au moment de la décision. La **cible**, souvent notée `y`, est la réponse que le modèle doit apprendre à produire.

### Tableau, ligne et colonne

Dans le code, pandas stocke les données dans un **DataFrame** : un tableau avec des lignes et des colonnes. Une ligne représente une observation ; une colonne représente une variable. Cette structure ressemble à une feuille Excel, mais elle peut être filtrée et transformée par Python.



## Le rôle et le type des colonnes

![Exemple de ligne avec identifiant features et cible](../../ressources/illustrations/jour_01/01_types_colonnes.png)

Une colonne peut être **numérique** comme une température, **catégorielle** comme un état de fonctionnement, ou **binaire** lorsqu’elle ne possède que deux réponses possibles. Son format ne suffit toutefois pas à décider si elle doit être utilisée.

Un identifiant peut être écrit avec des chiffres tout en ne représentant aucune grandeur mesurable. Le donner au modèle peut l’inciter à reconnaître les machines déjà vues au lieu d’apprendre des relations générales.

Avant de construire `X` et `y`, on pose donc trois questions :

1. cette information sera-t-elle disponible au moment de prédire ?
2. décrit-elle réellement la situation observée ?
3. contient-elle directement ou indirectement la réponse future ?

Une réponse négative ou ambiguë doit déclencher une vérification avant l’entraînement.


![Séparation des mesures en features X et cible y](../../ressources/illustrations/jour_01/01_features_cible.png)

*Les features décrivent la situation connue ; la cible représente la réponse historique à apprendre.*


## Classification, régression ou apprentissage non supervisé ?

- La **classification** prédit une catégorie : panne ou pas de panne, type d'objet, présence ou absence d'un casque.
- La **régression** prédit un nombre continu : température future, durée restante ou consommation.
- L'**apprentissage non supervisé** ne dispose pas de cible connue et cherche une structure, par exemple des groupes ou des observations atypiques.

Reconnaître un chiffre ou une catégorie d’image est un problème de classification ; prévoir une consommation ou une durée est un problème de régression.


![Comparaison classification régression et non supervisé](../../ressources/illustrations/jour_01/01_types_problemes.png)

*Le type de réponse attendue détermine la famille du problème avant le choix du modèle.*


## Le type de problème vient de la réponse attendue

On ne choisit pas d’abord un algorithme. On commence par définir ce que devra produire le système.

- « panne ou pas de panne » possède deux classes : c’est une **classification binaire** ;
- « type de panne A, B ou C » possède plusieurs classes : c’est une **classification multiclasse** ;
- « température dans une heure » est une valeur mesurable : c’est une **régression** ;
- « quels groupes de machines se ressemblent ? » n’impose pas de bonne réponse connue : c’est un problème **non supervisé**.

En apprentissage supervisé, chaque ligne d’entraînement doit être associée à une cible fiable. Un label faux ou ambigu enseigne au modèle une mauvaise réponse, même si le code est correct.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_course_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ressources" / "donnees" / "maintenance_machines.csv").exists():
            return candidate
    raise FileNotFoundError("Racine du cours introuvable. Ouvrez le notebook depuis le dossier du cours.")


ROOT = find_course_root()
DATA_FILE = ROOT / "ressources" / "donnees" / "maintenance_machines.csv"
plt.style.use("seaborn-v0_8-whitegrid")

df = pd.read_csv(DATA_FILE)
print(df.dtypes)
print("\nValeurs manquantes :", int(df.isna().sum().sum()))
df.describe().T


## Lire le profil des colonnes

`dtypes` indique le type stocké dans chaque colonne : entier, nombre décimal ou texte. `isna().sum()` compte les valeurs manquantes ; une valeur manquante est une information absente, pas un zéro.

`describe()` calcule quelques repères sur les colonnes numériques : nombre de valeurs, moyenne, écart-type, minimum, quartiles et maximum. Le `.T` transpose simplement le tableau pour placer une variable par ligne et faciliter la lecture.

Ces statistiques servent à repérer une unité inattendue, une colonne constante ou une valeur très éloignée. Elles ne décident pas automatiquement qu’une valeur est fausse : il faut la confronter au sens métier.


## Construire la matrice X et le vecteur y

`X` est un tableau avec une ligne par observation et une colonne par feature. `y` contient une seule réponse par ligne. L'identifiant de machine n'est pas une mesure physique utile ici : le donner au modèle pourrait l'inciter à mémoriser des identifiants au lieu d'apprendre des relations générales.

La cible ne doit jamais être incluse dans `X`. Sinon, le modèle recevrait directement la réponse : ce serait une **fuite de cible**.


In [ ]:
features = [
    "temperature_c", "vibration_mm_s", "pressure_bar",
    "load_pct", "age_years", "days_since_maintenance",
]
target = "failure_7d"
X = df[features]
y = df[target]
print("Forme de X :", X.shape)
print("Forme de y :", y.shape)
print("Taux de panne :", f"{y.mean():.1%}")


## Une vraie démonstration de régression

La classification répond par une **classe**. La régression répond par un **nombre continu**. Nous allons demander à une régression linéaire d'estimer la température d'une machine à partir de sa charge et de son âge.

Une **régression linéaire** cherche une relation simple : elle combine les variables d'entrée avec des coefficients appris. Elle sert ici de premier repère, pas de vérité universelle.


![Droite de régression et résidus](../../ressources/illustrations/jour_01/01_regression_residus.png)

Une régression linéaire calcule une prédiction en additionnant une valeur de départ, appelée **interception** ou biais, et une contribution pour chaque feature. Chaque contribution dépend d’un **coefficient appris**.

On peut lire l’idée ainsi : `prédiction = base + coefficient × charge + coefficient × âge`. Les coefficients ne sont pas choisis à la main ; `fit` les ajuste sur les exemples d’entraînement. `predict` applique ensuite la relation apprise à de nouvelles lignes.

La ligne orange du schéma représente la relation moyenne apprise. Les points ne tombent pas tous exactement dessus, car deux features ne suffisent pas à expliquer toute la température. La distance entre un point réel et sa prédiction est le **résidu**.


### Pourquoi séparer aussi cette mini-régression ?

Le modèle apprend sur `X_reg_train` et `y_reg_train`. Les 25 % placés dans `X_reg_test` et `y_reg_test` restent invisibles pendant `fit` et servent à vérifier les prédictions.

`test_size=0.25` fixe cette proportion et `random_state=42` rend le tirage reproductible. Le notebook suivant expliquera en détail pourquoi un projet complet ajoute aussi un ensemble de validation.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_regression = df[["load_pct", "age_years"]]
y_regression = df["temperature_c"]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_regression, y_regression, test_size=0.25, random_state=42
)

regression = LinearRegression()
regression.fit(X_reg_train, y_reg_train)
temperature_predite = regression.predict(X_reg_test)

mae = mean_absolute_error(y_reg_test, temperature_predite)
rmse = np.sqrt(mean_squared_error(y_reg_test, temperature_predite))
print("MAE :", round(mae, 2), "°C")
print("RMSE :", round(rmse, 2), "°C")

plt.figure(figsize=(6, 5))
plt.scatter(y_reg_test, temperature_predite, alpha=0.45)
bornes = [
    min(y_reg_test.min(), temperature_predite.min()),
    max(y_reg_test.max(), temperature_predite.max()),
]
plt.plot(bornes, bornes, "--", color="black", label="prédiction parfaite")
plt.xlabel("Température réelle (°C)")
plt.ylabel("Température prédite (°C)")
plt.title("Régression : valeurs réelles et prédites")
plt.legend()
plt.show()


### Comment lire le résultat ?

L'écart entre la valeur réelle et la valeur prédite s'appelle une **erreur**, ou un **résidu**.

- La **MAE** est l'erreur absolue moyenne. Une MAE de 2 °C signifie qu'en moyenne les prédictions s'écartent d'environ 2 °C.
- La **RMSE** pénalise davantage les grosses erreurs. Si elle est nettement supérieure à la MAE, quelques prédictions sont probablement très mauvaises.
- Sur le graphique, plus un point est proche de la diagonale, plus la prédiction est juste.

Une métrique ne suffit jamais : il faut aussi regarder les erreurs concrètes et vérifier si elles sont acceptables pour l'usage prévu.


### MAE et RMSE avec un exemple simple

Imaginons trois erreurs absolues de `1`, `2` et `6` °C. La MAE les moyenne directement : la grosse erreur compte pour 6. La RMSE met d’abord les erreurs au carré : l’erreur de 6 °C influence donc beaucoup plus le résultat.

La MAE est souvent facile à expliquer dans l’unité métier. La RMSE attire davantage l’attention sur les grandes erreurs. Aucune des deux ne dit à elle seule pourquoi le modèle se trompe.

Enfin, une relation statistique n’est pas nécessairement une cause. Si charge et température évoluent ensemble, cela ne prouve pas que la charge soit la seule cause de la température. Le modèle décrit les données disponibles ; l’expertise métier reste nécessaire pour interpréter.


## Le déséquilibre des classes

Les pannes sont heureusement moins fréquentes que les situations normales. Un modèle qui répond toujours « pas de panne » peut donc obtenir une exactitude élevée tout en étant inutile. Cette observation justifiera l'usage de plusieurs métriques et d'une baseline.


## À vous de jouer — choisir les bonnes colonnes

Créez `X_exercice` avec uniquement vibration, température et jours depuis maintenance, puis `y_exercice` avec la cible. Affichez leurs formes.


In [ ]:
X_exercice = df[["vibration_mm_s", "temperature_c", "days_since_maintenance"]]
y_exercice = df["failure_7d"]
print(X_exercice.shape, y_exercice.shape)


**À retenir :** le modèle ne doit recevoir que des informations réellement disponibles au moment où l'on souhaite prédire.
